In [0]:
WITH state_emissions AS (
  SELECT
    state_abbr,
    SUM(TRY_CAST(REPLACE(`GHG emissions mtons CO2e`, ',', '') AS DOUBLE)) AS total_emissions
  FROM
    emissions_data
  GROUP BY
    state_abbr
),
top_10_states AS (
  SELECT
    state_abbr,
    total_emissions
  FROM
    state_emissions
  ORDER BY
    total_emissions DESC
  LIMIT 10
),
top_10_total AS (
  SELECT SUM(total_emissions) AS top_10_emissions
  FROM top_10_states
),
country_total AS (
  SELECT SUM(total_emissions) AS country_emissions
  FROM state_emissions
)
SELECT
  t.state_abbr,
  t.total_emissions,
  ROUND((t.total_emissions / c.country_emissions) * 100, 2) AS pct_of_country,
  ROUND((t10.top_10_emissions / c.country_emissions) * 100, 2) AS top_10_pct_of_country
FROM
  top_10_states t
  CROSS JOIN country_total c
  CROSS JOIN top_10_total t10
ORDER BY
  t.total_emissions DESC